# 🏨 ¿El modelo sabe la respuesta o la descubre?
## Modelos Supervisados y No Supervisados con datos de hotel

---

En esta actividad vas a trabajar con un dataset de reservas de hotel y vas a entrenar **dos tipos de modelos** en Python:

- 🟢 **Predices** si una reserva será cancelada usando un **árbol de decisión** (supervisado)
- 🟠 **Descubres** perfiles de huéspedes usando **K-Means** (no supervisado)
- 📊 **Evalúas** los resultados con métricas reales
- 💬 **Reflexionas** sobre cuándo usar cada tipo de modelo

**⏱ Duración:** 35 minutos | **🐍 Lenguaje:** Python | **☁️ Entorno:** Google Colab

> **Cómo usar este notebook:** Ejecuta cada celda en orden haciendo clic en el botón ▶ o pulsando `Shift + Enter`. Lee las instrucciones entre celdas antes de ejecutar.

---
## 🧠 Antes de empezar: ¿cuál es la diferencia?

| | 🟢 Supervisado | 🟠 No Supervisado |
|---|---|---|
| **¿Tienes etiquetas?** | Sí. Sabes la respuesta correcta. | No. El modelo descubre solo. |
| **¿Qué hace?** | Aprende a predecir una respuesta conocida. | Agrupa datos por similitud. |
| **Ejemplo en hotel** | Predecir si la reserva se cancelará. | Descubrir perfiles de huéspedes. |
| **Algoritmo hoy** | Decision Tree | K-Means |
| **Pregunta clave** | ¿Ya sabemos lo que buscamos? | ¿Qué patrones esconde el dato? |

---

## PARTE 1 · Generar y explorar el dataset
**⏱ 5 minutos**

Vamos a generar el dataset de reservas de hotel directamente en este notebook. Contiene **300 reservas** con información realista sobre el tipo de hotel, canal de reserva, precio, perfil del huésped y si la reserva fue cancelada o no.

In [ ]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

tipos_hotel     = ['Resort', 'City Hotel']
meses           = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
                   'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']
paises          = ['España','Francia','Portugal','Alemania','Italia','Reino Unido','USA','Brasil']
canales         = ['Directo','OTA','Agencia','Corporativo']
tipos_habitacion= ['Individual','Doble','Suite','Familiar']

rows = []
for i in range(300):
    tipo          = random.choice(tipos_hotel)
    mes           = random.choice(meses)
    noches        = random.randint(1, 14)
    adultos       = random.randint(1, 4)
    ninos         = random.randint(0, 2)
    pais          = random.choice(paises)
    canal         = random.choice(canales)
    habitacion    = random.choice(tipos_habitacion)
    precio_noche  = round(random.uniform(50, 400), 2)
    precio_total  = round(precio_noche * noches, 2)
    solicitudes   = random.randint(0, 5)
    reservas_prev = random.randint(0, 10)
    anticipacion  = random.randint(0, 365)

    prob = 0.20
    if canal == 'OTA':         prob += 0.20
    if anticipacion > 180:     prob += 0.15
    if noches > 7:             prob += 0.10
    if solicitudes == 0:       prob += 0.05
    if reservas_prev > 3:      prob -= 0.10
    cancelada = 1 if random.random() < prob else 0

    rows.append([i+1, tipo, mes, noches, adultos, ninos, pais, canal,
                 habitacion, precio_noche, precio_total,
                 solicitudes, reservas_prev, anticipacion, cancelada])

columnas = ['id','tipo_hotel','mes_llegada','noches','adultos','ninos',
            'pais_origen','canal_reserva','tipo_habitacion','precio_noche',
            'precio_total','solicitudes_especiales','reservas_previas',
            'dias_anticipacion','cancelada']

df = pd.DataFrame(rows, columns=columnas)
print(f'✅ Dataset generado: {df.shape[0]} filas · {df.shape[1]} columnas')
df.head()

In [ ]:
# Exploración básica del dataset
print('=== TIPOS DE DATOS ===')
print(df.dtypes)

print('\n=== ESTADÍSTICAS NUMÉRICAS ===')
print(df.describe().round(2))

print('\n=== DISTRIBUCIÓN DE CANCELACIONES ===')
conteo = df['cancelada'].value_counts()
print(f"No canceladas (0): {conteo[0]} reservas ({conteo[0]/len(df)*100:.1f}%)")
print(f"Canceladas    (1): {conteo[1]} reservas ({conteo[1]/len(df)*100:.1f}%)")

### 💬 Reflexión 1
> **¿Qué porcentaje de reservas fueron canceladas? ¿Te parece alto o bajo para un hotel real? ¿Qué columnas crees que influyen más en la cancelación?**
>
> *(Escribe tu respuesta aquí haciendo doble clic en esta celda)*

---

## PARTE 2 · Modelo Supervisado — Predecir cancelaciones 🟢
**⏱ 12 minutos**

**La pregunta:** Con los datos de la reserva (canal, precio, anticipación, historial...) ¿podemos predecir si esa reserva se va a cancelar *antes de que ocurra*?

Aquí **sí tenemos la respuesta correcta** (columna `cancelada`). El modelo aprende de los ejemplos del pasado para predecir el futuro.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# --- Seleccionar features y target ---
features = ['noches', 'adultos', 'ninos', 'precio_noche',
            'solicitudes_especiales', 'reservas_previas', 'dias_anticipacion',
            'tipo_hotel', 'canal_reserva']

X = df[features].copy()
y = df['cancelada']

# Convertir columnas de texto a números (one-hot encoding)
X = pd.get_dummies(X, columns=['tipo_hotel', 'canal_reserva'])

print('Features finales:')
for col in X.columns:
    print(f'  · {col}')
print(f'\nForma de X: {X.shape}')

In [ ]:
# --- Dividir en entrenamiento (80%) y prueba (20%) ---
# Nunca evaluamos el modelo con los mismos datos con los que lo entrenamos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Datos de entrenamiento: {X_train.shape[0]} reservas')
print(f'Datos de prueba:        {X_test.shape[0]} reservas')

In [ ]:
# --- Entrenar el árbol de decisión ---
modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)

# Predecir sobre el conjunto de prueba
y_pred = modelo.predict(X_test)

print('✅ Modelo entrenado y predicciones generadas')

### 📊 Métricas del modelo supervisado

| Métrica | Qué mide | Fórmula simplificada |
|---|---|---|
| **Accuracy** | Del total, ¿cuántas predicciones fueron correctas? | (TP + TN) / Total |
| **Precision** | De las que predije como canceladas, ¿cuántas lo fueron realmente? | TP / (TP + FP) |
| **Recall** | De las que sí se cancelaron, ¿cuántas detecté? | TP / (TP + FN) |
| **F1-Score** | Equilibrio entre Precision y Recall | 2 · (P · R) / (P + R) |

> **TP** = Verdadero Positivo · **TN** = Verdadero Negativo · **FP** = Falso Positivo · **FN** = Falso Negativo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# --- Métricas individuales ---
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print('=== MÉTRICAS DEL MODELO ===')
print(f'  Accuracy  : {accuracy:.2%}')
print(f'  Precision : {precision:.2%}')
print(f'  Recall    : {recall:.2%}')
print(f'  F1-Score  : {f1:.2%}')

print('\n=== REPORTE COMPLETO ===')
print(classification_report(y_test, y_pred,
      target_names=['No cancelada', 'Cancelada']))

In [ ]:
# --- Visualizar la matriz de confusión ---
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No cancela', 'Pred: Cancela'],
            yticklabels=['Real: No cancela', 'Real: Cancela'],
            ax=axes[0], linewidths=1)
axes[0].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Valor Real')
axes[0].set_xlabel('Valor Predicho')

# Importancia de features
importancias = pd.Series(
    modelo.feature_importances_, index=X.columns
).sort_values(ascending=True).tail(10)

colors = ['#2196F3' if v < importancias.max() * 0.5 else '#1565C0' for v in importancias]
importancias.plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('Importancia de Features (Top 10)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importancia')
axes[1].axvline(x=0, color='gray', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('supervisado_resultados.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Gráfico guardado como supervisado_resultados.png')

### 💬 Reflexión 2
> **Responde estas preguntas:**
> 1. ¿Cuál es el Accuracy de tu modelo? ¿Te parece suficiente para usarlo en producción?
> 2. ¿Qué es más peligroso para el hotel: un **Falso Positivo** (predecir cancelación que no ocurre) o un **Falso Negativo** (no predecir una cancelación que sí ocurre)? ¿Por qué?
> 3. ¿Qué feature fue la más importante para predecir cancelaciones?
>
> *(Escribe tu respuesta aquí haciendo doble clic en esta celda)*

---

## PARTE 3 · Modelo No Supervisado — Descubrir perfiles de huéspedes 🟠
**⏱ 12 minutos**

**La pregunta:** Sin saber nada de antemano, ¿podemos agrupar automáticamente a los huéspedes en perfiles según su comportamiento?

Ahora **olvidamos la columna `cancelada`**. El modelo no tiene respuestas correctas — busca grupos por similitud.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --- Features para clustering (solo numéricas) ---
features_cluster = ['noches', 'adultos', 'ninos', 'precio_noche',
                    'precio_total', 'solicitudes_especiales',
                    'reservas_previas', 'dias_anticipacion']

X_cluster = df[features_cluster].copy()

# Normalizar: K-Means es sensible a la escala
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print('✅ Datos normalizados y listos para clustering')
print(f'Forma: {X_scaled.shape}')

### 📐 Método del Codo (Elbow Method)

Antes de aplicar K-Means necesitamos decidir **cuántos clusters usar**. El método del codo nos ayuda: calculamos la **inercia** (suma de distancias al centro de cada cluster) para distintos valores de K. El punto donde la curva "dobla" como un codo es el número óptimo de clusters.

También usaremos el **Silhouette Score**: mide qué tan bien separados están los clusters entre sí (valores entre -1 y 1, más alto es mejor).

In [ ]:
# --- Calcular inercia y silhouette para k=2 hasta k=8 ---
inercias    = []
silhouettes = []
rango_k     = range(2, 9)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

# --- Visualizar ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico del codo
axes[0].plot(rango_k, inercias, 'o-', color='#E65100', linewidth=2, markersize=8)
axes[0].fill_between(rango_k, inercias, alpha=0.1, color='#E65100')
axes[0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inercia', fontsize=12)
axes[0].set_title('Método del Codo', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=3, color='gray', linestyle='--', alpha=0.7, label='K=3 elegido')
axes[0].legend()

# Silhouette Score
colors_sil = ['#4CAF50' if s == max(silhouettes) else '#90A4AE' for s in silhouettes]
axes[1].bar(rango_k, silhouettes, color=colors_sil, edgecolor='white')
axes[1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score por K', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for i, (k, s) in enumerate(zip(rango_k, silhouettes)):
    axes[1].text(k, s + 0.002, f'{s:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== MÉTRICAS POR K ===')
for k, inercia, sil in zip(rango_k, inercias, silhouettes):
    marca = ' ← mayor silhouette' if sil == max(silhouettes) else ''
    print(f'  K={k} · Inercia: {inercia:8.1f} · Silhouette: {sil:.3f}{marca}')

In [ ]:
# --- Aplicar K-Means con 3 clusters ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_scaled)

df['cluster'] = kmeans.labels_

print('=== TAMAÑO DE CADA CLUSTER ===')
for cluster_id, count in df['cluster'].value_counts().sort_index().items():
    pct = count / len(df) * 100
    print(f'  Cluster {cluster_id}: {count} huéspedes ({pct:.1f}%)')

print(f'\n=== MÉTRICAS DEL CLUSTERING ===')
sil_final = silhouette_score(X_scaled, kmeans.labels_)
print(f'  Inercia final    : {kmeans.inertia_:.1f}')
print(f'  Silhouette Score : {sil_final:.3f} (entre -1 y 1, más alto = mejor separación)')

In [ ]:
# --- Perfil promedio de cada cluster ---
perfil = df.groupby('cluster')[features_cluster].mean().round(2)

print('=== PERFIL PROMEDIO DE CADA CLUSTER ===')
print(perfil.T.to_string())

print('\n=== CANAL MÁS FRECUENTE POR CLUSTER ===')
print(df.groupby('cluster')['canal_reserva']
        .agg(lambda x: x.value_counts().index[0]))

print('\n=== PAÍS MÁS FRECUENTE POR CLUSTER ===')
print(df.groupby('cluster')['pais_origen']
        .agg(lambda x: x.value_counts().index[0]))

print('\n=== TASA DE CANCELACIÓN POR CLUSTER ===')
tasa = df.groupby('cluster')['cancelada'].mean()
for c, t in tasa.items():
    print(f'  Cluster {c}: {t:.1%} de cancelaciones')

In [ ]:
# --- Visualizar los clusters ---
colores = ['#2196F3', '#FF9800', '#4CAF50']
nombres = ['Cluster 0', 'Cluster 1', 'Cluster 2']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Anticipación vs Precio total
for i in range(3):
    grupo = df[df['cluster'] == i]
    axes[0].scatter(
        grupo['dias_anticipacion'], grupo['precio_total'],
        c=colores[i], label=nombres[i], alpha=0.6, s=60, edgecolors='white', linewidth=0.5
    )
axes[0].set_xlabel('Días de anticipación', fontsize=12)
axes[0].set_ylabel('Precio total (€)', fontsize=12)
axes[0].set_title('Clusters: Anticipación vs Precio', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Noches vs Precio por noche
for i in range(3):
    grupo = df[df['cluster'] == i]
    axes[1].scatter(
        grupo['noches'], grupo['precio_noche'],
        c=colores[i], label=nombres[i], alpha=0.6, s=60, edgecolors='white', linewidth=0.5
    )
axes[1].set_xlabel('Noches de estancia', fontsize=12)
axes[1].set_ylabel('Precio por noche (€)', fontsize=12)
axes[1].set_title('Clusters: Noches vs Precio/noche', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('clusters_huespedes.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Gráfico guardado como clusters_huespedes.png')

### 💬 Reflexión 3
> **Mira el perfil promedio de cada cluster y los gráficos. Responde:**
> 1. ¿Qué nombre le pondrías a cada cluster? Ejemplo: *'Viajero corporativo'*, *'Familia en verano'*, *'Turista espontáneo'*...
> 2. El Silhouette Score que obtuviste, ¿indica que los clusters están bien separados?
> 3. ¿Qué decisiones de negocio podría tomar el hotel con esta información?
>
> *(Escribe tu respuesta aquí haciendo doble clic en esta celda)*

---

## PARTE 4 · Comparación y reflexión final
**⏱ 6 minutos**

Acabas de usar el mismo dataset de dos formas completamente diferentes.

In [ ]:
# --- Resumen comparativo de métricas ---
print('=' * 55)
print('       RESUMEN COMPARATIVO DE MÉTRICAS')
print('=' * 55)

print('\n🟢 MODELO SUPERVISADO (Decision Tree)')
print(f'   Accuracy   : {accuracy:.2%}')
print(f'   Precision  : {precision:.2%}')
print(f'   Recall     : {recall:.2%}')
print(f'   F1-Score   : {f1:.2%}')

print('\n🟠 MODELO NO SUPERVISADO (K-Means, K=3)')
print(f'   Inercia          : {kmeans.inertia_:.1f}')
print(f'   Silhouette Score : {sil_final:.3f}')
for cluster_id, count in df['cluster'].value_counts().sort_index().items():
    print(f'   Cluster {cluster_id}        : {count} huéspedes ({count/len(df)*100:.1f}%)')

print('\n' + '=' * 55)
print('\n📋 TABLA COMPARATIVA')
comparativa = pd.DataFrame({
    'Característica': [
        '¿Necesita etiquetas?', 'Algoritmo', 'Objetivo',
        'Métrica principal', '¿Cuándo usarlo?'
    ],
    'Supervisado': [
        'Sí (columna cancelada)', 'Decision Tree',
        'Predecir cancelaciones', f'Accuracy: {accuracy:.2%}',
        'Cuando tienes datos históricos etiquetados'
    ],
    'No Supervisado': [
        'No', 'K-Means',
        'Descubrir perfiles', f'Silhouette: {sil_final:.3f}',
        'Cuando quieres descubrir patrones ocultos'
    ]
})
print(comparativa.to_string(index=False))

### 💬 Reflexión Final

> **1. ¿En qué situación real del hotel usarías el modelo supervisado? Pon un ejemplo concreto.**
>
> *(Escribe aquí)*

> **2. ¿En qué situación real usarías el modelo no supervisado? ¿Qué decisiones te ayudaría a tomar?**
>
> *(Escribe aquí)*

> **3. ¿Qué perdiste al no usar la columna `cancelada` en el clustering? ¿Qué ganaste?**
>
> *(Escribe aquí)*

> **4. Si tuvieras que explicarle la diferencia entre los dos modelos a alguien sin conocimientos técnicos, ¿cómo lo harías en 2 frases?**
>
> *(Escribe aquí)*

---

## ✅ ¡Actividad completada!

Has entrenado dos tipos de modelos sobre el mismo dataset y has visto en la práctica la diferencia entre **predecir** y **descubrir**. Esa distinción es la base de cualquier proyecto de Machine Learning real.

| | Hiciste esto hoy |
|---|---|
| 🟢 Supervisado | Predijiste cancelaciones con Decision Tree y evaluaste con Accuracy, Precision, Recall y F1 |
| 🟠 No Supervisado | Descubriste perfiles de huéspedes con K-Means y elegiste K con el método del codo y Silhouette Score |